[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/39_ppo_loss_solution.ipynb)

# 🔴 Solution: PPO (Proximal Policy Optimization) Clipped Loss

*RLHF & Preference Losses · Hard*

Reference implementation. Try it yourself in `39_ppo_loss.ipynb` first.

---
Implement PPO's **clipped surrogate objective**.

$$L^{\text{CLIP}} = \mathbb{E}\Big[\min\big(r_t A_t,\ \
\text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t\big)\Big],
\qquad r_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}$$

Return the **loss**, i.e. $-L^{\text{CLIP}}$, averaged over all elements
(honouring an optional mask).

### Signature
```python
def ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2, mask=None):
    ...  # -> scalar loss
```

`new_logps`, `old_logps`, `advantages` are `(batch, seq)` (or any matching
shape). `mask` is 1 for real tokens, 0 for padding.

### Rules
- Compute the ratio in log space: `exp(new - old)`, never `exp(new)/exp(old)`
- The `min` is taken **after** multiplying by the advantage, on the two signed
  products
- Return a scalar; with a mask, average over unmasked elements only
- Do not use any RL library

### Why min() and not just clip()
This is the question interviewers actually ask. Clipping alone is **not**
conservative — it is the `min` that makes the bound pessimistic, and the two
sides behave asymmetrically:

- **$A > 0$** (action was better than expected). The objective wants $r$ up. The
  clip caps the reward at $r = 1+\epsilon$, so past that there is **no gradient**
  — you cannot keep pushing a good action arbitrarily far in one update.
- **$A < 0$** (action was worse). The objective wants $r$ down. Now $rA$ becomes
  *more* negative as $r$ grows, and `min` selects that unclipped branch. So for
  a bad action that has become *more* likely, the gradient is **not** clipped —
  PPO always retains the ability to fix a mistake.

With `clip` alone and no `min`, that second case would also flatten out, leaving
a policy that had drifted badly with no gradient to recover. The `min` deliberately
keeps the penalty live in exactly the direction where you want it live.

### The trap
$\pi_{\text{old}}$ is frozen when the rollouts are collected, so on the very
first gradient step of an update $\theta = \theta_{\text{old}}$, every $r_t$ is
exactly 1, nothing clips, and the loss is just $-\bar{A}$. Clipping only starts
biting as later minibatches and inner epochs push $\theta$ away. Two
consequences worth saying out loud: a fresh-rollout loss that is not
$-\bar{A}$ is a bug in your ratio or your sign, and if you only ever take one
inner epoch, the clip is dead code and PPO degenerates to vanilla policy
gradient.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2, mask=None):
    # Probability ratio, formed in log space so nothing overflows.
    ratio = jnp.exp(new_logps - old_logps)

    unclipped = ratio * advantages
    clipped = jnp.clip(ratio, 1.0 - clip_ratio, 1.0 + clip_ratio) * advantages

    # Pessimistic bound: take the WORSE of the two signed products.
    surrogate = jnp.minimum(unclipped, clipped)

    if mask is None:
        return -jnp.mean(surrogate)

    mask = mask.astype(surrogate.dtype)
    return -jnp.sum(surrogate * mask) / jnp.maximum(jnp.sum(mask), 1.0)

In [ ]:
# 🔍 Verify
import jax.numpy as jnp

old = jnp.zeros((1, 5))
adv = jnp.array([[1.0, 1.0, 1.0, -1.0, -1.0]])

for delta in (0.0, 0.1, 0.5, 2.0):
    new = old + delta
    r = float(jnp.exp(delta))
    print(f"ratio={r:6.2f}  loss={float(ppo_loss(new, old, adv)):+.4f}")
# Positive-advantage terms stop improving past r = 1.2;
# negative-advantage terms keep getting punished.

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("ppo_loss")